In [1]:
# =========================================================
# CELL 1
# IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import math

In [2]:
# =========================================================
# CELL 2
# LOAD CSV
# =========================================================

df = pd.read_csv("input_parameter_robot_bipedal.csv")

# menampilkan 5 data pertama
print(df.head())

   thigh_mm  tibia_mm  foot_mm  mass_robot_kg  hip_angle_deg  knee_angle_deg  \
0        65        65       55            2.6             10              30   
1        65        65       55            2.6             10              30   
2        65        65       55            2.6             10              30   
3        65        65       55            2.6             10              45   
4        65        65       55            2.6             10              45   

   ankle_angle_deg  
0                0  
1               10  
2               20  
3                0  
4               10  


In [3]:
# =========================================================
# CELL 3
# PARAMETER KONSTANTA
# =========================================================

g = 9.81                # gravitasi
servo_torque = 0.98     # torque MG996R (N.m)

In [4]:
# =========================================================
# CELL 4
# LIST HASIL ANALISIS
# =========================================================

fk_x_list = []
fk_z_list = []

workspace_list = []

torque_list = []

com_list = []

inertia_list = []

stability_list = []

In [5]:
# =========================================================
# CELL 5
# FORWARD KINEMATICS
# =========================================================
#
# Persamaan:
#
# x = l1*cos(theta1)
#   + l2*cos(theta1+theta2)
#
# z = l1*sin(theta1)
#   + l2*sin(theta1+theta2)
#
# =========================================================

for index, row in df.iterrows():

    # panjang link
    l1 = row['thigh_mm'] / 1000
    l2 = row['tibia_mm'] / 1000

    # sudut joint
    theta1 = math.radians(row['hip_angle_deg'])
    theta2 = math.radians(row['knee_angle_deg'])

    # forward kinematics
    x = (l1 * math.cos(theta1)) + \
        (l2 * math.cos(theta1 + theta2))

    z = (l1 * math.sin(theta1)) + \
        (l2 * math.sin(theta1 + theta2))

    fk_x_list.append(x)
    fk_z_list.append(z)

print("Forward Kinematics selesai")

Forward Kinematics selesai


In [6]:
# =========================================================
# CELL 6
# WORKSPACE ANALYSIS
# =========================================================
#
# Persamaan:
#
# r_max = l1 + l2
#
# =========================================================

for index, row in df.iterrows():

    l1 = row['thigh_mm'] / 1000
    l2 = row['tibia_mm'] / 1000

    workspace = l1 + l2

    workspace_list.append(workspace)

print("Workspace selesai")

Workspace selesai


In [7]:
# =========================================================
# CELL 7
# TORQUE ANALYSIS
# =========================================================
#
# Persamaan:
#
# tau = m*g*d
#
# =========================================================

for index, row in df.iterrows():

    mass_robot = row['mass_robot_kg']

    l1 = row['thigh_mm'] / 1000
    l2 = row['tibia_mm'] / 1000

    # asumsi satu kaki menopang setengah massa
    m = mass_robot / 2

    # jarak COM sederhana
    d = (l1 + l2) / 2

    torque = m * g * d

    torque_list.append(torque)

print("Torque selesai")

Torque selesai


In [8]:
# =========================================================
# CELL 8
# CENTER OF MASS (COM)
# =========================================================
#
# Persamaan:
#
# COM = (Σm*x)/(Σm)
#
# =========================================================

for index, row in df.iterrows():

    l1 = row['thigh_mm'] / 1000
    l2 = row['tibia_mm'] / 1000
    lf = row['foot_mm'] / 1000

    com = ((l1/2) + (l2/2) + (lf/2)) / 3

    com_list.append(com)

print("COM selesai")

COM selesai


In [9]:
# =========================================================
# CELL 9
# INERSIA
# =========================================================
#
# Persamaan:
#
# I = (1/3)*m*l^2
#
# =========================================================

for index, row in df.iterrows():

    mass_robot = row['mass_robot_kg']

    l1 = row['thigh_mm'] / 1000
    l2 = row['tibia_mm'] / 1000

    m = mass_robot / 2

    inertia = (1/3) * m * ((l1 + l2) ** 2)

    inertia_list.append(inertia)

print("Inersia selesai")

Inersia selesai


In [10]:
# =========================================================
# CELL 10
# OPTIMASI LINK / STABILITAS
# =========================================================
#
# kondisi:
# torque < 60% kemampuan servo
#
# =========================================================

for torque in torque_list:

    if torque < (0.6 * servo_torque):
        stability = "STABLE"
    else:
        stability = "NOT_STABLE"

    stability_list.append(stability)

print("Optimasi link selesai")

Optimasi link selesai


In [11]:
# =========================================================
# CELL 11
# MASUKKAN HASIL KE DATAFRAME
# =========================================================

df['FK_X_m'] = fk_x_list
df['FK_Z_m'] = fk_z_list

df['Workspace_m'] = workspace_list

df['Torque_Nm'] = torque_list

df['COM_m'] = com_list

df['Inertia'] = inertia_list

df['Stability'] = stability_list

print(df.head())

   thigh_mm  tibia_mm  foot_mm  mass_robot_kg  hip_angle_deg  knee_angle_deg  \
0        65        65       55            2.6             10              30   
1        65        65       55            2.6             10              30   
2        65        65       55            2.6             10              30   
3        65        65       55            2.6             10              45   
4        65        65       55            2.6             10              45   

   ankle_angle_deg    FK_X_m    FK_Z_m  Workspace_m  Torque_Nm     COM_m  \
0                0  0.113805  0.053068         0.13   0.828945  0.030833   
1               10  0.113805  0.053068         0.13   0.828945  0.030833   
2               20  0.113805  0.053068         0.13   0.828945  0.030833   
3                0  0.101295  0.064532         0.13   0.828945  0.030833   
4               10  0.101295  0.064532         0.13   0.828945  0.030833   

    Inertia   Stability  
0  0.007323  NOT_STABLE  
1  0.00732

In [12]:
# =========================================================
# CELL 12
# SIMPAN HASIL ANALISIS
# =========================================================

df.to_csv("hasil_analisis_robot_bipedal.csv", index=False)

print("File berhasil disimpan")

File berhasil disimpan
